# 09o - EVA02 ViT-B/16 Vehicle ReID on CityFlowV2

Kaggle training notebook for EVA02 ViT-B/16 fine-tuning on CityFlowV2 using PK sampling, a BNNeck classifier head, AdamW, center loss, multi-GPU DataParallel, and chunked Market-1501 evaluation.

In [ ]:
import gc
import json
import os
import random
import shutil
import subprocess
import sys
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


pip_install("timm>=1.0.0", "gdown", "opencv-python-headless")

import cv2
import gdown
import torch

gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
runtime_summary = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device_count": torch.cuda.device_count(),
    "gpus": gpu_names,
}
print(json.dumps(runtime_summary, indent=2))

WORK_DIR = Path("/kaggle/working")
OUTPUT_DIR = WORK_DIR / "09o_eva02_vit_reid"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
HISTORY_PATH = OUTPUT_DIR / "training_history.json"
METRICS_PATH = OUTPUT_DIR / "final_metrics.json"
FINAL_MODEL_PATH = WORK_DIR / "eva02_vit_cityflowv2_final.pth"
LAST_MODEL_PATH = CHECKPOINT_DIR / "eva02_vit_cityflowv2_last.pth"
for directory in [OUTPUT_DIR, CHECKPOINT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CITYFLOW_DIR = Path("/tmp/cityflowv2")
RAW_CROP_DIR = Path("/tmp/cityflowv2_raw_crops")
REID_ROOT = Path("/tmp/cityflowv2_crops")
GDRIVE_ID = "13wNJpS_Oaoe-7y5Dzexg_Ol7bKu1OWuC"
ARCHIVE_NAME = "AIC22_Track1_MTMC_Tracking.zip"
ALLOWED_SPLITS = {"train", "validation"}
MAX_CROPS_PER_ID_CAM = 20
MIN_AREA = 2000
MIN_BBOX_SIDE = 30
QUERY_CAMS_PER_ID = 2
QUERY_FRAMES_PER_CAM = 10
IMG_SIZE = (256, 256)
BATCH_SIZE = 64
EVAL_BATCH_SIZE = 32
EVAL_CHUNK_SIZE = 1024
P_IDS = 16
K_INSTANCES = 4
NUM_WORKERS = 4
EPOCHS = 120
FREEZE_BACKBONE_EPOCHS = 5
CHECKPOINT_EPOCHS = [40, 80, 120]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE.startswith("cuda")
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def split_counts(root: Path) -> dict[str, int]:
    counts = {}
    for split in ["train", "query", "gallery"]:
        split_dir = root / split
        counts[split] = sum(1 for _ in split_dir.rglob("*.jpg")) if split_dir.exists() else 0
    return counts


def prepared_reid_exists(root: Path) -> bool:
    counts = split_counts(root)
    return all(counts[split] > 0 for split in ["train", "query", "gallery"])


def find_gt(cam_dir: Path) -> Path | None:
    direct = cam_dir / "gt.txt"
    if direct.exists():
        return direct
    nested = cam_dir / "gt" / "gt.txt"
    if nested.exists():
        return nested
    matches = list(cam_dir.rglob("gt.txt"))
    return matches[0] if matches else None


def find_video(cam_dir: Path) -> Path | None:
    for ext in ["avi", "mp4"]:
        for stem in ["vdo", "video"]:
            candidate = cam_dir / f"{stem}.{ext}"
            if candidate.exists():
                return candidate
    matches = list(cam_dir.glob("*.avi")) + list(cam_dir.glob("*.mp4"))
    return matches[0] if matches else None


def ensure_cityflow_download() -> Path:
    global CITYFLOW_DIR

    preferred_roots = [
        CITYFLOW_DIR,
        Path("/kaggle/input/cityflowv2"),
        Path("/kaggle/input/aic22-track1-mtmc-tracking"),
    ]
    for check_dir in preferred_roots:
        if check_dir.exists() and next(check_dir.rglob("vdo.avi"), None) is not None:
            print(f"CityFlowV2 already available at {check_dir}")
            CITYFLOW_DIR = check_dir
            return check_dir

    CITYFLOW_DIR.mkdir(parents=True, exist_ok=True)
    archive_path = Path(f"/tmp/{ARCHIVE_NAME}")
    if not archive_path.exists():
        print(f"Downloading CityFlowV2 from Google Drive (id={GDRIVE_ID})...")
        gdown.download(f"https://drive.google.com/uc?id={GDRIVE_ID}", str(archive_path), quiet=False)
    else:
        print(f"Archive already downloaded: {archive_path}")

    staging = Path("/tmp/_aic22_staging")
    staging.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {archive_path} to {staging}...")
    with zipfile.ZipFile(str(archive_path), "r") as zf:
        zf.extractall(str(staging))

    if archive_path.exists():
        archive_path.unlink()
        print("Deleted archive to reclaim space.")

    moved = 0
    skipped_splits = set()
    for vdo_path in sorted(staging.rglob("vdo.avi")):
        cam_dir = vdo_path.parent
        scene_dir = cam_dir.parent
        split_dir = scene_dir.parent
        cam_name = cam_dir.name
        scene_name = scene_dir.name
        split_name = split_dir.name
        if not cam_name.startswith("c") or not scene_name.startswith("S"):
            continue
        if split_name not in ALLOWED_SPLITS:
            skipped_splits.add(split_name)
            continue
        flat_name = f"{scene_name}_{cam_name}"
        target = CITYFLOW_DIR / flat_name
        if not target.exists():
            shutil.move(str(cam_dir), str(target))
            moved += 1

    if moved == 0:
        for split_name in sorted(ALLOWED_SPLITS):
            split_dir = Path(f"/tmp/{split_name}")
            if not split_dir.exists():
                continue
            for vdo_path in sorted(split_dir.rglob("vdo.avi")):
                cam_dir = vdo_path.parent
                scene_dir = cam_dir.parent
                if not cam_dir.name.startswith("c") or not scene_dir.name.startswith("S"):
                    continue
                flat_name = f"{scene_dir.name}_{cam_dir.name}"
                target = CITYFLOW_DIR / flat_name
                if not target.exists():
                    shutil.move(str(cam_dir), str(target))
                    moved += 1

    if staging.exists():
        shutil.rmtree(staging, ignore_errors=True)
    for extra_dir in [Path("/tmp/train"), Path("/tmp/test"), Path("/tmp/validation")]:
        if extra_dir.exists():
            shutil.rmtree(extra_dir, ignore_errors=True)
    for extra_file in [Path("/tmp/ReadMe.txt"), Path("/tmp/list_cam.txt")]:
        if extra_file.exists():
            extra_file.unlink()
    for license_file in Path("/tmp").glob("Dataset License*"):
        license_file.unlink(missing_ok=True)
    for extra_dir in [Path("/tmp/cam_framenum"), Path("/tmp/cam_loc"), Path("/tmp/cam_timestamp"), Path("/tmp/eval")]:
        if extra_dir.exists():
            shutil.rmtree(extra_dir, ignore_errors=True)

    print(json.dumps({
        "cityflow_dir": str(CITYFLOW_DIR),
        "flattened_cameras": len([d for d in CITYFLOW_DIR.iterdir() if d.is_dir()]),
        "skipped_splits": sorted(skipped_splits),
    }, indent=2))
    return CITYFLOW_DIR


def collect_camera_assets(data_root: Path):
    cameras = []
    cam_gt_paths = {}
    cam_vid_paths = {}
    for cam_dir in sorted(data_root.iterdir()):
        if not cam_dir.is_dir() or "_c" not in cam_dir.name:
            continue
        gt = find_gt(cam_dir)
        vid = find_video(cam_dir)
        if gt is None or vid is None:
            continue
        cameras.append(cam_dir.name)
        cam_gt_paths[cam_dir.name] = gt
        cam_vid_paths[cam_dir.name] = vid
    if cameras:
        return cameras, cam_gt_paths, cam_vid_paths

    for vdo_path in sorted(data_root.rglob("vdo.avi")):
        cam_dir = vdo_path.parent
        if cam_dir.name.startswith("c") and cam_dir.parent.name.startswith("S"):
            split_name = cam_dir.parent.parent.name
            if split_name not in ALLOWED_SPLITS:
                continue
            flat_name = f"{cam_dir.parent.name}_{cam_dir.name}"
        else:
            continue
        gt = find_gt(cam_dir)
        if gt is None:
            continue
        cameras.append(flat_name)
        cam_gt_paths[flat_name] = gt
        cam_vid_paths[flat_name] = vdo_path

    cameras = sorted(set(cameras))
    if not cameras:
        raise RuntimeError(f"No cameras with GT+video found under {data_root}")
    return cameras, cam_gt_paths, cam_vid_paths


def load_gt(gt_path: Path):
    rows = []
    with gt_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame, tid = int(parts[0]), int(parts[1])
            x, y, w, h = map(int, parts[2:6])
            rows.append((frame, tid, x, y, w, h))
    return rows


def extract_crops_from_camera(cam_name: str, gt_file: Path, vid_file: Path, crop_dir: Path, max_crops: int, min_area: int):
    gt_rows = load_gt(gt_file)
    if not gt_rows:
        print(f"  {cam_name}: empty GT file")
        return {}

    id_dets = defaultdict(list)
    for frame, tid, x, y, w, h in gt_rows:
        id_dets[tid].append((frame, x, y, w, h))

    frame_to_dets = defaultdict(list)
    for tid, dets in id_dets.items():
        dets = sorted(dets, key=lambda item: item[0])
        if len(dets) <= max_crops:
            sampled = dets
        else:
            step = len(dets) / max_crops
            sampled = [dets[int(i * step)] for i in range(max_crops)]
        for frame, x, y, w, h in sampled:
            if w * h >= min_area and w >= MIN_BBOX_SIDE and h >= MIN_BBOX_SIDE:
                frame_to_dets[frame].append((tid, x, y, w, h))

    if not frame_to_dets:
        return {}

    crop_dir.mkdir(parents=True, exist_ok=True)
    crops = defaultdict(list)
    cap = cv2.VideoCapture(str(vid_file))
    current_frame = 0
    target_frames = sorted(frame_to_dets.keys())
    target_idx = 0

    while target_idx < len(target_frames):
        ok, img = cap.read()
        if not ok:
            break
        current_frame += 1
        if current_frame < target_frames[target_idx]:
            continue
        if current_frame > target_frames[target_idx]:
            while target_idx < len(target_frames) and target_frames[target_idx] < current_frame:
                target_idx += 1
            if target_idx >= len(target_frames) or current_frame != target_frames[target_idx]:
                continue

        height, width = img.shape[:2]
        for tid, x, y, w, h in frame_to_dets[current_frame]:
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(width, x + w), min(height, y + h)
            if x2 - x1 < MIN_BBOX_SIDE or y2 - y1 < MIN_BBOX_SIDE:
                continue
            crop = img[y1:y2, x1:x2]
            fname = f"{tid:04d}_{cam_name}_f{current_frame:06d}.jpg"
            fpath = crop_dir / fname
            cv2.imwrite(str(fpath), crop)
            crops[tid].append(str(fpath))
        target_idx += 1

    cap.release()
    del gt_rows, id_dets, frame_to_dets, target_frames
    gc.collect()
    print(f"  {cam_name}: {sum(len(paths) for paths in crops.values())} crops from {len(crops)} vehicles")
    return dict(crops)


def load_existing_raw_crops(raw_dir: Path):
    all_crops = defaultdict(lambda: defaultdict(list))
    for fpath in sorted(raw_dir.glob("*.jpg")):
        parts = fpath.stem.split("_")
        if len(parts) < 4:
            continue
        tid = int(parts[0])
        cam_name = parts[1] + "_" + parts[2]
        all_crops[tid][cam_name].append(str(fpath))
    return {tid: dict(cam_map) for tid, cam_map in all_crops.items()}


def link_or_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def frame_number_from_name(path: str) -> int:
    stem = Path(path).stem
    marker = stem.rsplit("_f", 1)
    return int(marker[1]) if len(marker) == 2 and marker[1].isdigit() else 0


def build_prepared_splits(all_crops, root: Path):
    if root.exists():
        shutil.rmtree(root, ignore_errors=True)
    for split in ["train", "query", "gallery"]:
        (root / split).mkdir(parents=True, exist_ok=True)

    split_stats = {split: 0 for split in ["train", "query", "gallery"]}
    for tid, cam_map in sorted(all_crops.items()):
        pid_dir_name = f"ID_{tid:04d}"
        sorted_cam_items = sorted((cam_name, sorted(paths, key=frame_number_from_name)) for cam_name, paths in cam_map.items())
        eval_cam_names = [cam_name for cam_name, _ in sorted_cam_items[: max(1, min(len(sorted_cam_items), QUERY_CAMS_PER_ID))]]

        for cam_name, paths in sorted_cam_items:
            for path_str in paths:
                src = Path(path_str)
                dst = root / "train" / pid_dir_name / src.name
                link_or_copy(src, dst)
                split_stats["train"] += 1

            query_count = 0
            if cam_name in eval_cam_names and len(paths) > 1:
                query_count = min(QUERY_FRAMES_PER_CAM, len(paths) - 1)
            query_paths = paths[:query_count]
            gallery_paths = paths[query_count:]

            for path_str in query_paths:
                src = Path(path_str)
                dst = root / "query" / pid_dir_name / src.name
                link_or_copy(src, dst)
                split_stats["query"] += 1
            for path_str in gallery_paths:
                src = Path(path_str)
                dst = root / "gallery" / pid_dir_name / src.name
                link_or_copy(src, dst)
                split_stats["gallery"] += 1

    return split_stats


if prepared_reid_exists(REID_ROOT):
    prep_stats = split_counts(REID_ROOT)
    print(f"Reusing prepared ReID splits from {REID_ROOT}")
else:
    data_root = ensure_cityflow_download()
    cameras, cam_gt_paths, cam_vid_paths = collect_camera_assets(data_root)
    print(f"Using {len(cameras)} cameras with GT+video")

    existing_raw_count = sum(1 for _ in RAW_CROP_DIR.glob("*.jpg")) if RAW_CROP_DIR.exists() else 0
    if existing_raw_count > 100:
        print(f"Reusing {existing_raw_count} raw crops from {RAW_CROP_DIR}")
        all_crops = load_existing_raw_crops(RAW_CROP_DIR)
    else:
        RAW_CROP_DIR.mkdir(parents=True, exist_ok=True)
        all_crops = defaultdict(lambda: defaultdict(list))
        print("Extracting crops from videos...")
        for cam_name in cameras:
            cam_crops = extract_crops_from_camera(
                cam_name,
                cam_gt_paths[cam_name],
                cam_vid_paths[cam_name],
                RAW_CROP_DIR,
                MAX_CROPS_PER_ID_CAM,
                MIN_AREA,
            )
            for tid, paths in cam_crops.items():
                all_crops[tid][cam_name].extend(paths)
            del cam_crops
            gc.collect()
        all_crops = {tid: dict(cam_map) for tid, cam_map in all_crops.items()}

    prep_stats = build_prepared_splits(all_crops, REID_ROOT)

archive_path = Path(f"/tmp/{ARCHIVE_NAME}")
if archive_path.exists():
    archive_path.unlink()
if RAW_CROP_DIR.exists():
    shutil.rmtree(RAW_CROP_DIR, ignore_errors=True)
if CITYFLOW_DIR.exists() and str(CITYFLOW_DIR).startswith("/tmp/"):
    shutil.rmtree(CITYFLOW_DIR, ignore_errors=True)

for name in [
    "all_crops",
    "cam_crops",
    "cam_gt_paths",
    "cam_vid_paths",
    "cameras",
    "data_root",
    "existing_raw_count",
]:
    globals().pop(name, None)
prep_gc_collected = gc.collect()
if DEVICE.startswith("cuda"):
    torch.cuda.empty_cache()

print(json.dumps({
    "reid_root": str(REID_ROOT),
    "split_counts": prep_stats,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "eval_chunk_size": EVAL_CHUNK_SIZE,
    "epochs": EPOCHS,
    "checkpoint_epochs": CHECKPOINT_EPOCHS,
    "device": DEVICE,
    "prep_gc_collected": prep_gc_collected,
}, indent=2))

In [ ]:
import json
import random
from collections import defaultdict
from pathlib import Path

from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import transforms as T

CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD = [0.26862954, 0.26130258, 0.27577711]
INTERP = T.InterpolationMode.BICUBIC


def parse_prepared_cityflow(root: Path):
    train, query, gallery = [], [], []
    for split_name, split_list in [("train", train), ("query", query), ("gallery", gallery)]:
        split_dir = root / split_name
        if not split_dir.is_dir():
            raise FileNotFoundError(f"Prepared CityFlowV2 split not found: {split_dir}")
        for pid_dir in sorted(split_dir.iterdir()):
            if not pid_dir.is_dir() or not pid_dir.name.startswith("ID_"):
                continue
            for img_path in sorted(pid_dir.rglob("*.jpg")):
                parts = img_path.stem.split("_")
                if len(parts) < 4:
                    continue
                pid = int(parts[0])
                cam_name = parts[1] + "_" + parts[2]
                split_list.append((str(img_path), pid, cam_name))

    all_cams = sorted({cam for _, _, cam in train + query + gallery})
    cam2id = {cam: idx for idx, cam in enumerate(all_cams)}
    train = [(path, pid, cam2id[cam]) for path, pid, cam in train]
    query = [(path, pid, cam2id[cam]) for path, pid, cam in query]
    gallery = [(path, pid, cam2id[cam]) for path, pid, cam in gallery]

    train_pids = sorted({pid for _, pid, _ in train})
    pid2label = {pid: label for label, pid in enumerate(train_pids)}
    train = [(path, pid2label[pid], cam) for path, pid, cam in train]
    return train, query, gallery, cam2id


class ReIDDataset(Dataset):
    def __init__(self, items, transform=None):
        self.items = list(items)
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        path, pid, camid = self.items[index]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, pid, camid, path


class PKSampler(Sampler[int]):
    def __init__(self, data_source, p=16, k=4):
        self.data_source = list(data_source)
        self.num_pids_per_batch = p
        self.num_instances = k
        self.batch_size = p * k
        self.index_dic = defaultdict(list)
        for index, (_, pid, _) in enumerate(self.data_source):
            self.index_dic[pid].append(index)
        self.pids = sorted(self.index_dic.keys())
        self.length = max(1, len(self.pids) // self.num_pids_per_batch) * self.batch_size

    def __iter__(self):
        final_indices = []
        available_pids = self.pids.copy()
        random.shuffle(available_pids)
        while len(available_pids) >= self.num_pids_per_batch:
            selected_pids = available_pids[: self.num_pids_per_batch]
            available_pids = available_pids[self.num_pids_per_batch :]
            for pid in selected_pids:
                candidate_indices = self.index_dic[pid]
                if len(candidate_indices) >= self.num_instances:
                    final_indices.extend(random.sample(candidate_indices, self.num_instances))
                else:
                    final_indices.extend(np.random.choice(candidate_indices, size=self.num_instances, replace=True).tolist())
        return iter(final_indices[: self.length])

    def __len__(self):
        return self.length


train_tf = T.Compose([
    T.Resize(IMG_SIZE, interpolation=INTERP),
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(10),
    T.RandomCrop(IMG_SIZE),
    T.ColorJitter(brightness=0.2, contrast=0.15, saturation=0.1, hue=0.0),
    T.ToTensor(),
    T.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    T.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value="random"),
])
test_tf = T.Compose([
    T.Resize(IMG_SIZE, interpolation=INTERP),
    T.ToTensor(),
    T.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])

train_data, query_data, gallery_data, cam2id = parse_prepared_cityflow(REID_ROOT)
NUM_CLASSES = len({pid for _, pid, _ in train_data})
NUM_CAMERAS = len(cam2id)
assert NUM_CLASSES > 0, "Train split is empty"
assert len(query_data) > 0 and len(gallery_data) > 0, "Query/gallery split is empty"
assert BATCH_SIZE == P_IDS * K_INSTANCES, "Batch size must match P x K"

train_ds = ReIDDataset(train_data, train_tf)
query_ds = ReIDDataset(query_data, test_tf)
gallery_ds = ReIDDataset(gallery_data, test_tf)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=PKSampler(train_data, p=P_IDS, k=K_INSTANCES),
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
    drop_last=True,
    persistent_workers=NUM_WORKERS > 0,
)
query_loader = DataLoader(
    query_ds,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
    persistent_workers=NUM_WORKERS > 0,
)
gallery_loader = DataLoader(
    gallery_ds,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
    persistent_workers=NUM_WORKERS > 0,
)

print(json.dumps({
    "train_images": len(train_data),
    "query_images": len(query_data),
    "gallery_images": len(gallery_data),
    "num_classes": NUM_CLASSES,
    "num_cameras": NUM_CAMERAS,
    "train_batches": len(train_loader),
    "query_batches": len(query_loader),
    "gallery_batches": len(gallery_loader),
    "batch_size": BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "num_workers": NUM_WORKERS,
}, indent=2))

In [ ]:
import copy
import json

import timm
import torch.nn as nn
import torch.nn.functional as F

REQUESTED_VIT_MODEL = "eva02_base_patch16_clip_224.merged2b"
FALLBACK_VIT_MODEL = "eva02_base_patch16_clip_224"
EMBED_DIM = 768
BACKBONE_LR = 1e-5
HEAD_LR = 5e-4
WEIGHT_DECAY = 5e-4
WARMUP_EPOCHS = 5
LABEL_SMOOTHING = 0.05
TRIPLET_WEIGHT = 0.3
CENTER_WEIGHT = 5e-4
CENTER_START_EPOCH = 15
EMA_DECAY = 0.9999


def create_eva02_backbone(model_name: str = REQUESTED_VIT_MODEL):
    try:
        backbone = timm.create_model(model_name, pretrained=True, num_classes=0, img_size=IMG_SIZE[0])
        return backbone, model_name
    except Exception as exc:
        if model_name != REQUESTED_VIT_MODEL:
            raise
        print(f"Requested model {model_name!r} unavailable in this timm build: {exc}")
        print(f"Falling back to {FALLBACK_VIT_MODEL!r} while keeping the same EVA02 CLIP training recipe.")
        backbone = timm.create_model(FALLBACK_VIT_MODEL, pretrained=True, num_classes=0, img_size=IMG_SIZE[0])
        return backbone, FALLBACK_VIT_MODEL


class EVA02ReID(nn.Module):
    def __init__(self, num_classes, embed_dim=768, model_name=REQUESTED_VIT_MODEL):
        super().__init__()
        self.vit, self.vit_model_name = create_eva02_backbone(model_name)
        self.feat_dim = int(getattr(self.vit, "num_features", getattr(self.vit, "embed_dim", embed_dim)))
        self.bn = nn.BatchNorm1d(self.feat_dim)
        self.bn.bias.requires_grad_(False)
        self.proj = nn.Identity() if embed_dim == self.feat_dim else nn.Linear(self.feat_dim, embed_dim, bias=False)
        self.cls_head = nn.Linear(embed_dim, num_classes, bias=False)
        if isinstance(self.proj, nn.Linear):
            nn.init.kaiming_normal_(self.proj.weight, mode="fan_out")
        nn.init.normal_(self.cls_head.weight, std=0.001)

    def forward(self, x):
        tokens = self.vit.forward_features(x)
        pooled = self.vit.forward_head(tokens, pre_logits=True)
        if pooled.ndim == 3:
            pooled = pooled[:, 0]
        global_feat = pooled
        bn_feat = self.bn(global_feat)
        proj_feat = self.proj(bn_feat)
        norm_feat = F.normalize(proj_feat, p=2, dim=1)
        logits = self.cls_head(proj_feat)
        if self.training:
            return logits, global_feat, norm_feat
        return norm_feat


class CrossEntropyLabelSmooth(nn.Module):
    def __init__(self, num_classes, epsilon=0.05):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon

    def forward(self, inputs, targets):
        log_probs = F.log_softmax(inputs.float(), dim=1)
        with torch.no_grad():
            one_hot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
            smooth = (1.0 - self.epsilon) * one_hot + self.epsilon / self.num_classes
        return (-smooth * log_probs).sum(dim=1).mean()


class SoftTripletLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.loss_fn = nn.SoftMarginLoss()

    def forward(self, feats, pids):
        feats = F.normalize(feats.float(), p=2, dim=1)
        distances = torch.cdist(feats, feats, p=2)
        same_id = pids.unsqueeze(0).eq(pids.unsqueeze(1))
        same_id.fill_diagonal_(False)
        diff_id = ~same_id
        diff_id.fill_diagonal_(False)
        hardest_pos = distances.masked_fill(~same_id, float("-inf")).max(dim=1).values
        hardest_neg = distances.masked_fill(~diff_id, float("inf")).min(dim=1).values
        valid = torch.isfinite(hardest_pos) & torch.isfinite(hardest_neg)
        if not torch.any(valid):
            return feats.sum() * 0.0
        return self.loss_fn(hardest_neg[valid] - hardest_pos[valid], torch.ones_like(hardest_neg[valid]))


class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim))

    def forward(self, feats, labels):
        feats = feats.float()
        centers_batch = self.centers[labels]
        diff = feats - centers_batch
        return (diff * diff).sum(dim=1).mean()


class ModelEMA:
    def __init__(self, model, decay=0.9999):
        self.ema = copy.deepcopy(model)
        self.ema.eval()
        self.decay = decay
        for parameter in self.ema.parameters():
            parameter.requires_grad_(False)

    def update(self, model):
        with torch.no_grad():
            for ema_param, param in zip(self.ema.parameters(), model.parameters()):
                ema_param.data.mul_(self.decay).add_(param.data, alpha=1.0 - self.decay)
            for ema_buffer, model_buffer in zip(self.ema.buffers(), model.buffers()):
                ema_buffer.copy_(model_buffer)


def unwrap_model(current_model):
    return current_model.module if hasattr(current_model, "module") else current_model


def set_backbone_trainable(current_model, trainable: bool):
    raw_model = unwrap_model(current_model)
    for parameter in raw_model.vit.parameters():
        parameter.requires_grad_(trainable)
    for parameter in raw_model.bn.parameters():
        parameter.requires_grad_(True)
    for parameter in raw_model.cls_head.parameters():
        parameter.requires_grad_(True)
    if isinstance(raw_model.proj, nn.Linear):
        for parameter in raw_model.proj.parameters():
            parameter.requires_grad_(True)


print(json.dumps({
    "requested_vit_model": REQUESTED_VIT_MODEL,
    "fallback_vit_model": FALLBACK_VIT_MODEL,
    "embed_dim": EMBED_DIM,
    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
    "weight_decay": WEIGHT_DECAY,
    "warmup_epochs": WARMUP_EPOCHS,
    "center_start_epoch": CENTER_START_EPOCH,
    "ema_decay": EMA_DECAY,
}, indent=2))

In [ ]:
import json
import math
import time

id_loss_fn = CrossEntropyLabelSmooth(NUM_CLASSES, epsilon=LABEL_SMOOTHING).to(DEVICE)
triplet_loss_fn = SoftTripletLoss().to(DEVICE)
center_loss_fn = CenterLoss(NUM_CLASSES, EMBED_DIM).to(DEVICE)

model = EVA02ReID(num_classes=NUM_CLASSES, embed_dim=EMBED_DIM, model_name=REQUESTED_VIT_MODEL).to(DEVICE)
SELECTED_VIT_MODEL = model.vit_model_name
NUM_GPUS = torch.cuda.device_count()
if NUM_GPUS > 1:
    print(f"Using DataParallel on {NUM_GPUS} GPUs")
    model = nn.DataParallel(model)
raw_model = unwrap_model(model)
ema = ModelEMA(raw_model, decay=EMA_DECAY)


def build_optimizers(current_model):
    raw = unwrap_model(current_model)
    head_params = []
    for name, parameter in raw.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith("vit."):
            continue
        head_params.append(parameter)
    optimizer = torch.optim.AdamW(
        [
            {"params": raw.vit.parameters(), "lr": BACKBONE_LR},
            {"params": head_params, "lr": HEAD_LR},
        ],
        betas=(0.9, 0.999),
        weight_decay=WEIGHT_DECAY,
    )
    center_optimizer = torch.optim.SGD(center_loss_fn.parameters(), lr=0.5)

    def lr_lambda(epoch_index: int) -> float:
        if epoch_index < WARMUP_EPOCHS:
            return float(epoch_index + 1) / float(max(WARMUP_EPOCHS, 1))
        progress = float(epoch_index - WARMUP_EPOCHS + 1) / float(max(EPOCHS - WARMUP_EPOCHS, 1))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    return optimizer, center_optimizer, scheduler, scaler


def epoch_checkpoint_path(epoch: int) -> Path:
    return CHECKPOINT_DIR / f"eva02_vit_cityflowv2_epoch_{epoch:03d}.pth"


def save_checkpoint(checkpoint_path: Path, epoch: int, map_value: float, state_dict=None):
    checkpoint_path = Path(checkpoint_path)
    payload = {
        "model": state_dict if state_dict is not None else unwrap_model(model).state_dict(),
        "epoch": int(epoch),
        "mAP": float(map_value),
    }
    torch.save(payload, checkpoint_path)
    return checkpoint_path


set_backbone_trainable(model, False)
optimizer, center_optimizer, scheduler, scaler = build_optimizers(model)


def train_one_epoch(epoch: int):
    raw = unwrap_model(model)
    if epoch == FREEZE_BACKBONE_EPOCHS + 1:
        set_backbone_trainable(model, True)
        print(f"Unfroze backbone at epoch {epoch}")

    model.train()
    use_center = epoch >= CENTER_START_EPOCH
    running = {"loss": 0.0, "id_loss": 0.0, "tri_loss": 0.0, "cen_loss": 0.0}

    for batch_idx, (imgs, pids, _camids, _paths) in enumerate(train_loader, start=1):
        imgs = imgs.to(DEVICE, non_blocking=USE_AMP)
        pids = pids.to(DEVICE, non_blocking=USE_AMP).long()

        optimizer.zero_grad(set_to_none=True)
        if use_center:
            center_optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            logits, global_feat, _norm_feat = model(imgs)
            loss_id = id_loss_fn(logits, pids)
            loss_tri = triplet_loss_fn(global_feat, pids)
            loss_cen_raw = center_loss_fn(global_feat.float(), pids) if use_center else torch.tensor(0.0, device=global_feat.device)
            loss = loss_id + (TRIPLET_WEIGHT * loss_tri) + (CENTER_WEIGHT * loss_cen_raw)

        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            if use_center:
                scaler.unscale_(center_optimizer)
            torch.nn.utils.clip_grad_norm_(raw.parameters(), max_norm=5.0)
            if use_center:
                for parameter in center_loss_fn.parameters():
                    if parameter.grad is not None:
                        parameter.grad.data *= 1.0 / max(CENTER_WEIGHT, 1e-12)
            scaler.step(optimizer)
            if use_center:
                scaler.step(center_optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(raw.parameters(), max_norm=5.0)
            if use_center:
                for parameter in center_loss_fn.parameters():
                    if parameter.grad is not None:
                        parameter.grad.data *= 1.0 / max(CENTER_WEIGHT, 1e-12)
            optimizer.step()
            if use_center:
                center_optimizer.step()

        ema.update(raw)
        running["loss"] += float(loss.detach().item())
        running["id_loss"] += float(loss_id.detach().item())
        running["tri_loss"] += float(loss_tri.detach().item())
        running["cen_loss"] += float(loss_cen_raw.detach().item()) if use_center else 0.0

        if batch_idx % 10 == 0 or batch_idx == len(train_loader):
            denom = float(batch_idx)
            print(
                f"Epoch {epoch:03d} | Batch {batch_idx:03d}/{len(train_loader):03d} | "
                f"loss={running['loss'] / denom:.4f} id={running['id_loss'] / denom:.4f} "
                f"tri={running['tri_loss'] / denom:.4f} cen={running['cen_loss'] / denom:.4f} "
                f"lr_backbone={optimizer.param_groups[0]['lr']:.2e} lr_head={optimizer.param_groups[1]['lr']:.2e}"
            )

    scheduler.step()
    denom = float(max(len(train_loader), 1))
    return {key: value / denom for key, value in running.items()}


history = []
saved_checkpoints = []
start_time = time.time()
for epoch in range(1, EPOCHS + 1):
    metrics = train_one_epoch(epoch)
    record = {
        "epoch": epoch,
        "lr_backbone": float(optimizer.param_groups[0]["lr"]),
        "lr_head": float(optimizer.param_groups[1]["lr"]),
        **{key: float(value) for key, value in metrics.items()},
    }
    history.append(record)
    with HISTORY_PATH.open("w", encoding="utf-8") as handle:
        json.dump(history, handle, indent=2)

    if epoch in CHECKPOINT_EPOCHS:
        checkpoint_path = save_checkpoint(epoch_checkpoint_path(epoch), epoch, 0.0, copy.deepcopy(ema.ema.state_dict()))
        saved_checkpoints.append(str(checkpoint_path))
        print(f"Saved checkpoint -> {checkpoint_path}")

last_checkpoint = save_checkpoint(LAST_MODEL_PATH, EPOCHS, 0.0, copy.deepcopy(ema.ema.state_dict()))
elapsed_hours = (time.time() - start_time) / 3600.0
print(json.dumps({
    "selected_vit_model": SELECTED_VIT_MODEL,
    "saved_checkpoints": saved_checkpoints,
    "last_checkpoint": str(last_checkpoint),
    "history_path": str(HISTORY_PATH),
    "elapsed_hours": round(elapsed_hours, 2),
}, indent=2))

In [ ]:
import gc
import json
from pathlib import Path

import numpy as np


@torch.no_grad()
def extract_features_chunked(current_model, loader, device="cuda", flip=True):
    current_model.eval()
    feat_chunks, pid_chunks, cam_chunks = [], [], []
    for imgs, pids, camids, _paths in loader:
        imgs = imgs.to(device, non_blocking=True)
        features = current_model(imgs)
        if flip:
            flipped = current_model(torch.flip(imgs, dims=[3]))
            features = F.normalize((features + flipped) / 2.0, p=2, dim=1)
        feat_chunks.append(features.cpu().numpy())
        pid_chunks.append(pids.numpy())
        cam_chunks.append(camids.numpy())
        del imgs, features
        if device.startswith("cuda"):
            torch.cuda.empty_cache()
    if not feat_chunks:
        return np.zeros((0, EMBED_DIM), dtype=np.float32), np.zeros(0, dtype=np.int64), np.zeros(0, dtype=np.int64)
    return np.concatenate(feat_chunks), np.concatenate(pid_chunks), np.concatenate(cam_chunks)


def eval_market1501_chunked(query_features, gallery_features, q_pids, g_pids, q_camids, g_camids, chunk_size=1024, max_rank=50):
    num_q = query_features.shape[0]
    num_g = gallery_features.shape[0]
    if num_q == 0 or num_g == 0:
        return 0.0, np.zeros(max_rank, dtype=np.float32)
    max_rank = min(max_rank, num_g)
    gallery_t = gallery_features.T
    all_cmc = []
    all_ap = []

    for start in range(0, num_q, chunk_size):
        end = min(start + chunk_size, num_q)
        q_chunk = query_features[start:end]
        q_pid_chunk = q_pids[start:end]
        q_cam_chunk = q_camids[start:end]
        dist_chunk = 1.0 - q_chunk @ gallery_t
        indices = np.argsort(dist_chunk, axis=1)
        matches = (g_pids[indices] == q_pid_chunk[:, None]).astype(np.int32)

        for row_index in range(end - start):
            q_pid = q_pid_chunk[row_index]
            q_cam = q_cam_chunk[row_index]
            order = indices[row_index]
            remove = (g_pids[order] == q_pid) & (g_camids[order] == q_cam)
            keep = ~remove
            raw_cmc = matches[row_index][keep]
            if raw_cmc.sum() == 0:
                continue
            cmc = raw_cmc.cumsum()
            cmc[cmc > 1] = 1
            cmc = cmc[:max_rank]
            if cmc.shape[0] < max_rank:
                pad_value = int(cmc[-1]) if cmc.shape[0] else 0
                cmc = np.pad(cmc, (0, max_rank - cmc.shape[0]), constant_values=pad_value)
            all_cmc.append(cmc)
            num_rel = raw_cmc.sum()
            tmp_cmc = raw_cmc.cumsum()
            precision = tmp_cmc / (np.arange(len(tmp_cmc)) + 1.0)
            all_ap.append(float((precision * raw_cmc).sum() / num_rel))

        del q_chunk, q_pid_chunk, q_cam_chunk, dist_chunk, indices, matches
        gc.collect()

    if not all_cmc:
        return 0.0, np.zeros(max_rank, dtype=np.float32)
    cmc = np.asarray(all_cmc, dtype=np.float32).mean(axis=0)
    mAP = float(np.mean(all_ap))
    return mAP, cmc


def build_eval_model(checkpoint_path: Path):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model_name = SELECTED_VIT_MODEL if "SELECTED_VIT_MODEL" in globals() else REQUESTED_VIT_MODEL
    eval_model = EVA02ReID(num_classes=NUM_CLASSES, embed_dim=EMBED_DIM, model_name=model_name)
    state_dict = checkpoint.get("model", checkpoint)
    missing, unexpected = eval_model.load_state_dict(state_dict, strict=True)
    assert not missing, missing
    assert not unexpected, unexpected
    return eval_model.to(DEVICE), checkpoint


def evaluate_checkpoint(checkpoint_path: Path):
    checkpoint_path = Path(checkpoint_path)
    eval_model, checkpoint = build_eval_model(checkpoint_path)
    if torch.cuda.device_count() > 1:
        eval_model = nn.DataParallel(eval_model)
    q_feats, q_pids, q_camids = extract_features_chunked(eval_model, query_loader, DEVICE, flip=True)
    g_feats, g_pids, g_camids = extract_features_chunked(eval_model, gallery_loader, DEVICE, flip=True)
    mAP, cmc = eval_market1501_chunked(
        q_feats,
        g_feats,
        q_pids,
        g_pids,
        q_camids,
        g_camids,
        chunk_size=EVAL_CHUNK_SIZE,
    )
    eval_model = eval_model.module if hasattr(eval_model, "module") else eval_model
    eval_model = eval_model.to("cpu")
    del eval_model, q_feats, q_pids, q_camids, g_feats, g_pids, g_camids
    gc.collect()
    if DEVICE.startswith("cuda"):
        torch.cuda.empty_cache()
    return {
        "checkpoint_path": str(checkpoint_path),
        "epoch": int(checkpoint.get("epoch", 0)),
        "mAP": float(mAP),
        "rank1": float(cmc[0]) if len(cmc) > 0 else 0.0,
        "rank5": float(cmc[4]) if len(cmc) > 4 else 0.0,
        "rank10": float(cmc[9]) if len(cmc) > 9 else 0.0,
    }


checkpoint_paths = [epoch_checkpoint_path(epoch) for epoch in CHECKPOINT_EPOCHS]
checkpoint_paths.append(LAST_MODEL_PATH)
checkpoint_paths = [path for path in checkpoint_paths if Path(path).exists()]
assert checkpoint_paths, "No checkpoints found to evaluate"

evaluation_results = [evaluate_checkpoint(path) for path in checkpoint_paths]
best_result = max(evaluation_results, key=lambda item: item["mAP"])
best_checkpoint = torch.load(best_result["checkpoint_path"], map_location="cpu", weights_only=False)
torch.save(
    {
        "model": best_checkpoint["model"],
        "epoch": int(best_result["epoch"]),
        "mAP": float(best_result["mAP"]),
    },
    FINAL_MODEL_PATH,
)

final_payload = {
    "requested_vit_model": REQUESTED_VIT_MODEL,
    "resolved_vit_model": SELECTED_VIT_MODEL if "SELECTED_VIT_MODEL" in globals() else REQUESTED_VIT_MODEL,
    "final_model_path": str(FINAL_MODEL_PATH),
    "evaluation_results": evaluation_results,
    "best_result": best_result,
    "config": {
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "eval_chunk_size": EVAL_CHUNK_SIZE,
        "p_ids": P_IDS,
        "k_instances": K_INSTANCES,
        "epochs": EPOCHS,
        "freeze_backbone_epochs": FREEZE_BACKBONE_EPOCHS,
        "checkpoint_epochs": CHECKPOINT_EPOCHS,
        "backbone_lr": BACKBONE_LR,
        "head_lr": HEAD_LR,
        "weight_decay": WEIGHT_DECAY,
        "warmup_epochs": WARMUP_EPOCHS,
        "label_smoothing": LABEL_SMOOTHING,
        "triplet_weight": TRIPLET_WEIGHT,
        "triplet_soft_margin": True,
        "center_weight": CENTER_WEIGHT,
        "center_start_epoch": CENTER_START_EPOCH,
        "ema_decay": EMA_DECAY,
    },
}
with METRICS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(final_payload, handle, indent=2)

print(json.dumps(final_payload, indent=2))